# Dual-Task Tile Recognition Challenge

**Introduction:** This challenge involves analyzing mixed images created from STL-10 dataset pairs to predict both the source image and original position of individual tiles.

## I. Problem Overview
The task uses images created by combining two different STL-10 images from different class labels. Each original image is resized to 96×96 pixels and split into a 3×3 grid of tiles (each 32×32 pixels). All 18 tiles from both images are then randomly shuffled and arranged into a 3×6 grid, creating mixed images for analysis.

## II. Dataset
The dataset consists of 2,400 mixed images for training and 329 mixed images for validation. Each mixed image contains 18 tiles from two different STL-10 source images. The data is split with images and labels prepared for immediate use. File sizes are optimized for quick training on moderate hardware or free Colab GPUs.

## III. Task
Design and implement a PyTorch neural network model to analyze the mixed images and predict two properties for every tile. The specific requirements are as follows:

1. **Source Classification**: Determine which original image the tile came from
  - Label 0: Tile from the first source image (brighter image)
  - Label 1: Tile from the second source image (darker image)

2. **Position Classification**: Identify the tile's original position in the 3×3 layout
  - Labels 0–8: Corresponding to positions in the original 3×3 grid

**Model Requirements:**
- Name the model class as `TileClassifier()`. Using other names may lead to evaluation issues.
- Include at least 3 convolutional layers (nn.Conv2d) with corresponding batch normalization and pooling layers.
- Include at least 2 fully connected layers (nn.Linear) for each classification head.
- Use appropriate activation functions such as nn.ReLU.
- The model should accept input tensors of shape [B×18, 3, 32, 32] where B is the batch size.
- Output two sets of logits: source_logits [B×18, 2] and position_logits [B×18, 9].
- The loss function, optimizer, and learning rate can be freely selected.

## IV. Evaluation
Performance is measured using accuracy metrics for both classification tasks:
- **Source Accuracy**: `correct_source_predictions / total_tiles`
- **Position Accuracy**: `correct_position_predictions / total_tiles`
- **Combined Score**: `(Source_Accuracy + Position_Accuracy) / 2`

In [1]:
import os
print("hi")

hi


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import time

class MixedSTL10Dataset(Dataset):
    def __init__(self, root_dir, split="train"):
        """
        Args:
            root_dir (string): Directory with all the images and labels.
            split (string): 'train' or 'val'.
        """
        self.img_dir = os.path.join(root_dir, split, "images")
        self.label_dir = os.path.join(root_dir, split, "labels")
        self.img_files = sorted(os.listdir(self.img_dir))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.img_files[idx])
        image = plt.imread(img_name)
        if image.dtype == np.uint8:
            image = image / 255.0
        image = torch.tensor(image).permute(2, 0, 1).float()

        # Load labels (which image and position for each tile)
        label_name = os.path.join(
            self.label_dir, os.path.splitext(self.img_files[idx])[0] + ".npy"
        )
        labels = np.load(label_name, allow_pickle=True)

        # Extract tiles and corresponding labels
        tiles = []
        tile_labels_source = []
        tile_labels_position = []

        for idx, (source, position) in enumerate(labels):
            # Calculate position in mixed image (3x6 grid)
            row = idx // 6
            col = idx % 6

            # Extract the tile (32x32)
            tile = image[:, row * 32 : (row + 1) * 32, col * 32 : (col + 1) * 32]
            tiles.append(tile)

            # Store labels
            tile_labels_source.append(source)
            tile_labels_position.append(position)

        return (
            torch.stack(tiles),  # Shape: [18, 3, 32, 32]
            torch.tensor(tile_labels_source),  # Shape: [18]
            torch.tensor(tile_labels_position),  # Shape: [18]
        )

In [3]:
# ==================================================================================
# COMPETITION ZONE: MODIFY CODE BELOW THIS LINE KEEPING METHOD AND CLASS INTERFACES
# ==================================================================================
# Some ideas:
# 1. Add global context features that consider relationships between all tiles rather than processing each independently
# 2. Experiment with different pooling strategies and feature aggregation methods throughout the network
# 3. Try multi-scale feature extraction approaches to capture information at different levels of detail
# 4. Modify how features are shared or separated between the two classification tasks within the model architecture
# 5. Adjust network depth, width, and connectivity patterns to better capture the hierarchical nature of visual features
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = channels // reduction
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(x)


class MBConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand=2, use_se=True):
        super().__init__()

        hidden = in_ch * expand

        self.use_res = (stride == 1 and in_ch == out_ch)

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),

            nn.Conv2d(hidden, hidden, 3, stride=stride, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),

            SEBlock(hidden) if use_se else nn.Identity(),

            nn.Conv2d(hidden, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_res else out


class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.blocks = nn.Sequential(
            MBConv(32, 32, stride=1, expand=2),
            MBConv(32, 32, stride=1, expand=2),

            MBConv(32, 64, stride=2, expand=2),
            MBConv(64, 64, stride=1, expand=2),

            MBConv(64, 128, stride=2, expand=2),
            MBConv(128, 128, stride=1, expand=2),
        )

        self.head = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

class TileClassifier(nn.Module):
    def __init__(self):
        super(TileClassifier, self).__init__()

        # Define the base CNN layers
        # self.cnn = nn.Sequential(
        #     nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
        #     nn.BatchNorm2d(32),
        #     nn.ReLU(True),

        #     nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
        #     nn.BatchNorm2d(32),
        #     nn.ReLU(True),

        #     nn.MaxPool2d(2),

        #     nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False),
        #     nn.Conv2d(32, 64, kernel_size=1, bias=False),
        #     nn.BatchNorm2d(64),
        #     nn.ReLU(True),

        #     nn.MaxPool2d(2),

        #     nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False),
        #     nn.Conv2d(64, 128, kernel_size=1, bias=False),
        #     nn.BatchNorm2d(128),
        #     nn.ReLU(True),

        #     nn.MaxPool2d(2),
        #     nn.Conv2d(128, 128, 3, padding=1, groups=128, bias=False),
        #     nn.Conv2d(128, 256, 1, bias=False),
        #     nn.BatchNorm2d(256), nn.ReLU(True),

        #     nn.AdaptiveAvgPool2d((1,1)),
        #     nn.Flatten()
        # )

        self.cnn = Model()

        self.source_classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 2)
        )
        self.position_classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 9)
        )

        self.attn = nn.MultiheadAttention(128, 4, batch_first=True)
        self.attn_norm = nn.LayerNorm(128)

    def forward(self, x):
        # x shape: [B * 18, 3, 32, 32], where B is batch_size from DataLoader
        B = x.shape[0] // 18

        tile_features = self.cnn(x)
        tile_features = tile_features.reshape(B, 18, 128)

        attn_out, _ = self.attn(tile_features, tile_features, tile_features)
        tile_features = self.attn_norm(torch.cat(tile_features, attn_out))
        tile_features = tile_features.reshape(B*18, 128)

        position_logits = self.position_classifier(tile_features)

        source_logits = self.source_classifier(tile_features)
        return source_logits, position_logits

def setup_training_components(device):
    lr = 1e-3

    model = TileClassifier().to(device)
    print("Model architecture:")
    print(model)

    criterion_source = nn.CrossEntropyLoss()
    criterion_position = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

    return model, criterion_source, criterion_position, optimizer, scheduler

# ==================================================================================
# DO NOT EDIT CODE BELOW THIS LINE!
# ==================================================================================

In [4]:
def get_device():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    return device

def get_data_loaders(args):
    train_dataset = MixedSTL10Dataset(root_dir=args.data_dir, split="train")
    val_dataset = MixedSTL10Dataset(root_dir=args.data_dir, split="val")

    print(f"Training set size: {len(train_dataset)}")
    print(f"Validation set size: {len(val_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader


def train_one_epoch(model, loader, crit_source, crit_pos, optimizer, device, sw, pw):
    model.train()
    total_loss, src_correct, pos_correct, total = 0.0, 0, 0, 0

    for tiles, src_labels, pos_labels in loader:
        batch_size = tiles.size(0)
        tiles = tiles.view(-1, 3, 32, 32).to(device)
        src_labels = src_labels.view(-1).to(device)
        pos_labels = pos_labels.view(-1).to(device)

        src_logits, pos_logits = model(tiles)
        src_loss = crit_source(src_logits, src_labels)
        pos_loss = crit_pos(pos_logits, pos_labels)
        loss = sw * src_loss + pw * pos_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_size
        src_correct += (src_logits.argmax(1) == src_labels).sum().item()
        pos_correct += (pos_logits.argmax(1) == pos_labels).sum().item()
        total += src_labels.size(0)

    return {
        "loss": total_loss / len(loader.dataset),
        "source_acc": src_correct / total,
        "position_acc": pos_correct / total,
    }

def evaluate(model, loader, crit_source, crit_pos, device, sw, pw):
    model.eval()
    total_loss, src_correct, pos_correct, total = 0.0, 0, 0, 0

    with torch.no_grad():
        for tiles, src_labels, pos_labels in loader:
            batch_size = tiles.size(0)
            tiles = tiles.view(-1, 3, 32, 32).to(device)
            src_labels = src_labels.view(-1).to(device)
            pos_labels = pos_labels.view(-1).to(device)

            src_logits, pos_logits = model(tiles)
            src_loss = crit_source(src_logits, src_labels)
            pos_loss = crit_pos(pos_logits, pos_labels)
            loss = sw * src_loss + pw * pos_loss

            total_loss += loss.item() * batch_size
            src_correct += (src_logits.argmax(1) == src_labels).sum().item()
            pos_correct += (pos_logits.argmax(1) == pos_labels).sum().item()
            total += src_labels.size(0)

    return {
        "loss": total_loss / len(loader.dataset),
        "source_acc": src_correct / total,
        "position_acc": pos_correct / total,
    }


def print_epoch_summary(epoch, total_epochs, start_time, train_metrics, val_metrics):
    print(f"Epoch {epoch + 1}/{total_epochs} | Time: {time.time() - start_time:.2f}s")
    print(
        f"Train Loss: {train_metrics['loss']:.4f} | Source Acc: {train_metrics['source_acc']:.4f} | "
        f"Position Acc: {train_metrics['position_acc']:.4f}"
    )
    print(
        f"Val Loss: {val_metrics['loss']:.4f} | Source Acc: {val_metrics['source_acc']:.4f} | "
        f"Position Acc: {val_metrics['position_acc']:.4f}"
    )

def save_model_checkpoint(model, optimizer, epoch, metrics, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": metrics["loss"],
        "val_source_acc": metrics["source_acc"],
        "val_position_acc": metrics["position_acc"],
    }, path)
    print(f"Saved new best model with validation loss: {metrics['loss']:.4f}")


def train_model(args):
    device = get_device()
    train_loader, val_loader = get_data_loaders(args)
    model, criterion_source, criterion_position, optimizer, scheduler = setup_training_components(device)

    best_val_loss = float("inf")
    os.makedirs(args.output_dir, exist_ok=True)
    best_model_path = os.path.join(args.output_dir, "best_model.pth")

    source_weight = 2.0
    position_weight = 1.0

    print("Starting training...")
    for epoch in range(args.epochs):
        start_time = time.time()

        train_metrics = train_one_epoch(
            model, train_loader, criterion_source, criterion_position, optimizer, device, source_weight, position_weight
        )

        val_metrics = evaluate(
            model, val_loader, criterion_source, criterion_position, device, source_weight, position_weight
        )

        scheduler.step(val_metrics['loss'])

        print_epoch_summary(epoch, args.epochs, start_time, train_metrics, val_metrics)

        # Uncomment to enable best model saving
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            save_model_checkpoint(model, optimizer, epoch, val_metrics, best_model_path)

    print(f"Training complete. Best validation loss: {best_val_loss:.4f}")
    return model

In [5]:
data_dir = "./mixed_dataset"
output_dir = "./model_output"
batch_size = 8
epochs = 30
# lr = 0.001
# kernel_size = 5
# num_layers = 3

# Create an args object with these parameters
class Args:
    def __init__(self):
        self.data_dir = data_dir
        self.output_dir = output_dir
        self.batch_size = batch_size
        self.epochs = epochs

args = Args()

In [6]:
train_model(args)

Using device: cuda
Training set size: 5000
Validation set size: 500
Model architecture:
TileClassifier(
  (cnn): Model(
    (stem): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (blocks): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (5): ReLU(inplace=True)
          (6): SEBlock(
            (fc): Sequential(
              (0): AdaptiveAvgPool2d(output_size=1)
              (1): Con

TypeError: cat() received an invalid combination of arguments - got (Tensor, Tensor), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)
